In [2]:
# Install required packages and load environment variables
%pip install anthropic
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()
client = Anthropic()
model = "claude-sonnet-4-6"

Note: you may need to restart the kernel to use updated packages.


In [3]:
# Helper functions â€” chat() now supports stop_sequences
def add_user_message(messages, text):
    messages.append({"role": "user", "content": text})

def add_assistant_message(messages, text):
    messages.append({"role": "assistant", "content": text})

def chat(messages, system=None, temperature=1.0, stop_sequences=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
    }
    if system:
        params["system"] = system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences
    message = client.messages.create(**params)
    return message.content[0].text

In [4]:
# Tanpa teknik apapun â€” Claude jawab dengan markdown formatting
messages = []
add_user_message(messages, "Generate a very short event bridge rule as json")

text = chat(messages)
text

'Here\'s a simple EventBridge rule in JSON:\n\n```json\n{\n  "source": ["aws.ec2"],\n  "detail-type": ["EC2 Instance State-change Notification"],\n  "detail": {\n    "state": ["running"]\n  }\n}\n```\n\nThis rule triggers when an **EC2 instance** changes state to **"running"**.'

In [5]:
# Teknik 1: System Prompt untuk kontrol format output
# Prefill tidak support di model Claude terbaru
# Alternatif: instruksikan lewat system prompt
messages = []
add_user_message(messages, "Generate a very short event bridge rule as json")

text = chat(messages, system="Respond with only raw valid JSON. No markdown, no explanation, no code blocks.")
text

'{\n  "source": ["aws.ec2"],\n  "detail-type": ["EC2 Instance State-change Notification"],\n  "detail": {\n    "state": ["running"]\n  }\n}'

In [6]:
# Teknik 2: Instruksi format di dalam user message
messages = []
add_user_message(messages, "Generate a very short event bridge rule as json. Return only raw JSON, no markdown.")

text = chat(messages)
text

'{\n  "source": ["aws.ec2"],\n  "detail-type": ["EC2 Instance State-change Notification"],\n  "detail": {\n    "state": ["running"]\n  }\n}'

In [ ]:
import json

messages = []
add_user_message(
	messages,
	"Generate a very short event bridge rule as json. Return only raw JSON, no markdown, no code blocks."
)

text = chat(
	messages,
	system="Output ONLY a JSON object. Start your response with { and end with }. Absolutely no other text. Do not explain anything. Do not provide any code."
)

data = json.loads(text.strip())
data


In [14]:
import json

json.loads(text.strip())  # Valid JSON, no markdown, no code blocks

{'source': ['aws.ec2'],
 'detail-type': ['EC2 Instance State-change Notification'],
 'detail': {'state': ['running']}}

In [21]:
masssages = []

prompt = """
Generate three different sample AWS CLI commands. Each should be very short.
"""

add_user_message(masssages, prompt)

text = chat(messages)
text.strip()

'{\n  "source": ["aws.ec2"],\n  "detail-type": ["EC2 Instance State-change Notification"],\n  "detail": {\n    "state": ["stopped"]\n  }\n}'

In [23]:
from IPython.display import Markdown, display

display(Markdown(text))  # Render the output as markdown in the notebook

{
  "source": ["aws.ec2"],
  "detail-type": ["EC2 Instance State-change Notification"],
  "detail": {
    "state": ["stopped"]
  }
}

In [31]:
#from IPython.display import Markdown, display
messages = []
prompt = """
"Give me three different AWS CLI commands. Each should be very short.
"""
add_user_message(messages, prompt)

# System prompt menggantikan prefill - paksa format langsung tanpa penjelasan
# Stop sequence output sebelum ada teks keempat

text = chat(
    messages,
    system="Here are all three commands in a single block without any comments, '```', explanations, or numbering. Do not add any text after the third command.",
    stop_sequences=["4."]
)

#display(Markdown(text))  # Output is exactly 3 commands, no explanations, no comments, no numbering, no blank lines
print(text)

aws s3 ls
aws ec2 describe-instances
aws iam list-users
